# Unsupervised Learning on PLAsTiCC Light-Curve Features

We take a table of features describing how astronomical objects vary in brightness and ask
an **unsupervised** question: *without telling the algorithm what these objects are, can it
rediscover the natural classes?* We walk the full lecture pipeline — **scale -> reduce
(PCA / UMAP) -> cluster (k-means, DBSCAN) -> validate (silhouette, ARI)** — and we look
closely at where it succeeds and where, and *why*, it breaks.

## What is this data?

**PLAsTiCC** — the *Photometric LSST Astronomical Time-series Classification Challenge*, run
on Kaggle in 2018 ([Zenodo record 2539456](https://zenodo.org/records/2539456)). It is a set
of **simulated light curves** for the Vera C. Rubin Observatory / LSST: each object is
observed repeatedly in six filters (passbands **u, g, r, i, z, y**), and its brightness
(*flux*) traces out a curve in time.

Different astrophysical objects vary in characteristically different ways — an exploding star
(supernova) rises and fades once; a pulsating star (RR Lyrae, Mira) repeats; an eclipsing
binary dips periodically. The science goal is to tell these classes apart.

Raw, irregularly-sampled time series are awkward, so each light curve has been compressed
into **summary features** — 11 statistics computed *per passband*:

> `amplitude`, `max_slope`, `maximum`, `minimum`, `median`, `weighted_average`,
> `median_absolute_deviation`, `std`, `skew`, `percent_beyond_1_std`, `percent_close_to_median`

That is 11 x 6 = 66 numbers, plus three per-object **metadata** columns — `z`, `zerr`
(redshift and its error) and `mwebv` (Milky-Way dust reddening) — for **69 features**
describing **7848 objects**. 

In [14]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from astropy.table import Table
import scipy.stats as spstat
from collections import OrderedDict

rng = np.random.RandomState(0)
datadir = 'data'


## Part 1 — Load the data and take a first look

The feature table is a structured array: one named column per object, plus `feature` and
`channel` rows. We unpack it into an `(n_objects, n_features)` matrix — the shape every
scikit-learn tool expects.

In [3]:
pbmap = {'0': 'u', '1': 'g', '2': 'r', '3': 'i', '4': 'z', '5': 'y', '': 'meta'}

raw = np.load(f'{datadir}/plasticc_featuretable.npz', allow_pickle=True)['features']
feature_names = np.array([f'{f}_{pbmap[c]}' for f, c in zip(raw['feature'], raw['channel'])])
object_ids = [int(c) for c in raw.dtype.names if c not in ('feature', 'channel')]

X_all = np.column_stack([raw[str(oid)] for oid in object_ids]).T   # (n_obj, 69)
print(f'all objects: {X_all.shape}   NaNs: {np.isnan(X_all).sum()}')

all objects: (7848, 69)   NaNs: 0


In [10]:
pd.DataFrame(X_all, columns=feature_names)

,amplitude_u,amplitude_g,amplitude_r,amplitude_i,amplitude_z,amplitude_y,percent_beyond_1_std_u,percent_beyond_1_std_g,percent_beyond_1_std_r,percent_beyond_1_std_i,...,std_r,std_i,std_z,std_y,weighted_average_u,weighted_average_g,weighted_average_r,weighted_average_i,weighted_average_z,weighted_average_y
0,121.048015,880.533203,646.921722,488.190826,402.069122,400.501617,0.476190,0.586207,0.568966,0.551724,...,451.180827,332.520885,289.276965,292.182295,-17.061118,-212.397193,-102.220639,-101.206639,-54.744845,-59.688379
1,14.622504,10.422385,10.298480,11.862455,11.057368,14.491025,0.385714,0.392857,0.410714,0.410714,...,5.718981,6.392561,6.349526,7.030448,-3.500958,-1.322397,-1.030469,-1.382941,-1.407879,-1.876399
2,4.701063,4.543094,11.921774,19.503951,23.498145,33.234935,0.333333,0.442308,0.134615,0.115385,...,5.505767,8.112835,10.604821,13.201397,-0.016423,-0.034170,2.059833,2.988513,4.486335,5.057690
3,10.944189,97.931352,111.477482,104.097369,99.563790,75.881339,0.180556,0.035714,0.071429,0.089286,...,31.671373,34.654080,32.772464,25.822133,1.176322,3.652226,6.716857,12.514694,12.247387,8.760515
4,6.067815,19.896143,54.378113,71.309338,80.071971,60.009062,0.333333,0.172414,0.137931,0.103448,...,21.135263,26.043193,26.633303,21.245772,0.824380,3.617169,7.842645,8.830427,8.463856,5.602845
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7843,37.357845,80.459251,22.959684,17.285638,40.893879,311.488880,0.117647,0.071429,0.083333,0.360000,...,9.575063,8.947735,16.471863,96.198290,2.330913,7.408239,2.910996,5.045740,-2.895793,12.631473
7844,147.049413,28.318704,8.846144,180.521552,203.362503,102.181900,0.083333,0.090909,0.500000,0.071429,...,4.476771,84.970426,106.131224,38.274181,11.993140,4.793174,-0.641505,17.704391,47.282142,2.299252
7845,93.877627,49.707789,54.212563,65.235669,43.384130,128.713696,0.230769,0.352941,0.416667,0.352941,...,29.227103,30.551444,22.011905,47.524380,-0.509159,-18.878684,-31.794794,-19.209125,-13.344091,-12.738886
7846,24.886840,163.773464,16.968529,17.940242,20.410165,131.960207,0.214286,0.090909,0.160000,0.480000,...,6.418259,7.512520,9.936292,43.614047,1.727450,5.458514,1.855982,-1.180270,0.211007,9.368137


### Drop the metadata columns

Three of the 69 columns are **not light-curve features** — they describe *where the object
sits*, not *how it varies*:

- `z`, `zerr` — **redshift**: how far away the object is.
- `mwebv` — foreground **dust** reddening from our own Galaxy, set by sky position.

We drop them for two reasons:

1. **They are a different kind of information.** We want to group objects by their
   *behaviour* (the shape of the light curve), not by distance or sky location.
2. **`z` is the answer in disguise.** Galactic objects (stars in our own Milky Way) sit at
   redshift exactly `0`; extragalactic objects do not. So `z` *is* essentially the
   galactic/extragalactic label. Feeding it to an unsupervised algorithm and then crediting
   the algorithm for separating the two would be circular — we would be handing it the answer
   we claim it discovered. This is **data leakage**.

In [ ]:
meta = pd.read_csv(f'{datadir}/plasticc_train_metadata.csv').set_index('object_id').loc[object_ids]
# the target column has the classes available here in Table 1: https://iopscience.iop.org/article/10.3847/1538-4365/accd6a/pdf

,ra,decl,ddf_bool,hostgal_specz,hostgal_photoz,hostgal_photoz_err,distmod,mwebv,target,true_target,...,true_rv,true_av,true_peakmjd,libid_cadence,tflux_u,tflux_g,tflux_r,tflux_i,tflux_z,tflux_y
object_id,,,,,,,,,,,,,,,,,,,,,
615,349.0461,-61.9438,1,0.000,0.000,0.000,-9.000,0.017,92,92,...,0.0,0.0,59570.000,69,484.7,3286.7,3214.1,3039.7,2854.5,2837.0
713,53.0859,-27.7844,1,1.818,1.627,0.255,45.406,0.007,88,88,...,0.0,0.0,59570.000,34,108.7,117.7,119.9,149.6,147.9,150.5
730,33.5742,-6.5796,1,0.232,0.226,0.016,40.256,0.021,42,42,...,0.0,0.0,60444.379,9,0.0,0.0,0.0,0.0,0.0,0.0
745,0.1899,-45.5867,1,0.304,0.281,1.152,40.795,0.007,90,90,...,0.0,0.0,60130.453,38,0.0,0.0,0.0,0.0,0.0,0.0
1124,352.7113,-63.8237,1,0.193,0.241,0.018,40.417,0.024,90,90,...,0.0,0.0,60452.641,1,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130739978,26.7188,-14.9403,0,0.000,0.000,0.000,-9.000,0.013,65,65,...,0.0,0.0,59570.000,18232,26.4,267.6,742.8,3295.1,6047.5,7955.2
130755807,120.1013,-62.6967,0,0.172,2.561,1.115,46.611,0.136,90,90,...,0.0,0.0,60056.809,14934,0.0,0.0,0.0,0.0,0.0,0.0
130762946,203.1081,-55.6821,0,0.000,0.000,0.000,-9.000,0.430,16,16,...,0.0,0.0,59570.000,47805,83.8,1124.7,1445.1,1191.2,848.9,382.5


In [ ]:
# Keep only genuine light-curve features
is_meta = np.isin(feature_names, [f'{m}_meta' for m in ('z', 'zerr', 'mwebv')])
X_all = X_all[:, ~is_meta]
feature_names = feature_names[~is_meta]
print(f'after dropping metadata: {X_all.shape}')

# Labels. Used ONLY for (a) selecting the galactic subset and (b) validation in Part 6.
# They are never fed to any clustering step.
meta = pd.read_csv(f'{datadir}/plasticc_train_metadata.csv').set_index('object_id').loc[object_ids]
# separate the galactic and extragalactic 
galactic = meta['hostgal_specz'].to_numpy() == 0.0
extragalactic = meta['hostgal_specz'].to_numpy() > 0
X = X_all[galactic]


after dropping metadata: (7848, 66)


,ra,decl,ddf_bool,hostgal_specz,hostgal_photoz,hostgal_photoz_err,distmod,mwebv,target,true_target,...,true_rv,true_av,true_peakmjd,libid_cadence,tflux_u,tflux_g,tflux_r,tflux_i,tflux_z,tflux_y
object_id,,,,,,,,,,,,,,,,,,,,,
615,349.0461,-61.9438,1,0.000,0.000,0.000,-9.000,0.017,92,92,...,0.0,0.0,59570.000,69,484.7,3286.7,3214.1,3039.7,2854.5,2837.0
713,53.0859,-27.7844,1,1.818,1.627,0.255,45.406,0.007,88,88,...,0.0,0.0,59570.000,34,108.7,117.7,119.9,149.6,147.9,150.5
730,33.5742,-6.5796,1,0.232,0.226,0.016,40.256,0.021,42,42,...,0.0,0.0,60444.379,9,0.0,0.0,0.0,0.0,0.0,0.0
745,0.1899,-45.5867,1,0.304,0.281,1.152,40.795,0.007,90,90,...,0.0,0.0,60130.453,38,0.0,0.0,0.0,0.0,0.0,0.0
1124,352.7113,-63.8237,1,0.193,0.241,0.018,40.417,0.024,90,90,...,0.0,0.0,60452.641,1,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130739978,26.7188,-14.9403,0,0.000,0.000,0.000,-9.000,0.013,65,65,...,0.0,0.0,59570.000,18232,26.4,267.6,742.8,3295.1,6047.5,7955.2
130755807,120.1013,-62.6967,0,0.172,2.561,1.115,46.611,0.136,90,90,...,0.0,0.0,60056.809,14934,0.0,0.0,0.0,0.0,0.0,0.0
130762946,203.1081,-55.6821,0,0.000,0.000,0.000,-9.000,0.430,16,16,...,0.0,0.0,59570.000,47805,83.8,1124.7,1445.1,1191.2,848.9,382.5


The data above is a description of the light curves shown below:

In [11]:
lightcurve_data = pd.read_csv('data/plasticc_train_lightcurves.csv')

## Part 2 — Scaling, and the distance metric

Clustering runs on **distance**. k-means asks *"which centroid is this point closest to?"*;
UMAP and DBSCAN ask *"which points sit near each other?"*. By default that distance is the
**Euclidean (L2)** metric from the lecture:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_j (a_j - b_j)^2}$$

Every feature `j` enters through its *raw numerical size*, so a feature with large units
silently dominates the distance regardless of how informative it is. That is why we scale.
Let's try the standard recipe — `StandardScaler` (subtract mean, divide by std) — and look
at what comes out.

In [4]:
from sklearn.preprocessing import StandardScaler

Xs_naive = StandardScaler().fit_transform(X)
print(f'raw feature range:      {X.min():,.0f}  to  {X.max():,.0f}')
print(f'after StandardScaler, largest |z-score| of any single value: {np.abs(Xs_naive).max():.1f}')

raw feature range:      -1,149,388  to  2,432,809
after StandardScaler, largest |z-score| of any single value: 48.1


The raw features span *millions*, and even after standardising, one value lands ~48 standard
deviations from the mean. A handful of extreme objects (very bright, or numerical outliers)
blow up the L2 distance: pairwise distances are dominated by those few coordinates, so every
object looks roughly *equidistant* from every other except the outliers — and k-means will
cheerfully spend a whole cluster on a single extreme point.

The fix matches the data. These are brightness-like features spanning **orders of
magnitude**, so we compress them with a **signed logarithm** before standardising. `log`
pulls in the long tail; the `sign` / `log1p` form handles zeros and negative values:

$$x \;\rightarrow\; \mathrm{sign}(x)\,\log(1 + |x|)$$

In [5]:
# signed-log compresses dynamic range, THEN standardise so every feature is comparable
X_log = np.sign(X) * np.log1p(np.abs(X))
Xs = StandardScaler().fit_transform(X_log)
print(f'after signed-log + StandardScaler, largest |z-score|: {np.abs(Xs).max():.1f}')
print('No single object dominates the distance metric any more.')

after signed-log + StandardScaler, largest |z-score|: 8.6
No single object dominates the distance metric any more.


## Part 3 — PCA and the scree plot

We have 66 features, many correlated (the six passbands of one statistic move together).
**PCA** finds the directions of greatest variance so we can describe the data with fewer
numbers. The **scree plot** shows how much variance each component captures; we keep just
enough to retain **95%** of the total.

## Part 4 — A non-linear embedding

PCA is **linear** — it can only rotate and stretch. Real class structure is often curved, so
we follow the lecture and pass the PCA output through **UMAP** (with **t-SNE** alongside for
comparison), which lay the data out in 2D while trying to preserve *local neighbourhoods*.

We plot the embedding **uncoloured first**, exactly as we would have to on real unlabelled
data — all we get is geometry.

**How many groups do you see?** Hold that guess — we test it against the truth at the very end. For now the algorithm
knows nothing about classes, and neither (officially) do we.

## Part 5 — Clustering: how many clusters?

`k`-means needs you to *choose* `k` up front, and no label is available to tell you the right
value. We use the **silhouette score** (lecture: cohesion vs. separation, range -1 to +1) as
an internal, label-free guide and sweep over `k`.

Notice the silhouette is **almost flat** — it hovers near 0.6 from `k=2` to `k=7` with no
decisive peak. Geometrically, several different `k` all carve up these islands about equally
well, and the score cannot tell us which carving is *meaningful*. This is the core limit of
an internal metric: it rewards *tidy* partitions, not *correct* ones.

So we make two reasonable choices and compare them in the next part:

- **`k = 3`** — roughly the number of distinct islands the eye picks out.
- **`k = 5`** — the number of named classes we are secretly hoping to recover.

We also run **DBSCAN**, which sets the number of clusters itself from local density rather
than taking a fixed `k`.

## Part 6 — Validation: bring in the labels

Everything above was **label-blind** — scaling, PCA, UMAP, k-means never saw `y_true`. Now we
use the labels, and it is worth being clear about *why that is allowed*:

- **We have labels here.** We are not clustering to *obtain* labels; we already have them. We
  look in order to **grade the method** — did unsupervised clustering recover known structure?
  The tool is the **Adjusted Rand Index (ARI)**: agreement between clusters and true classes,
  0 = chance, 1 = perfect.
- **On real unlabelled data, ARI is impossible.** You would have only silhouette and your
  eyes. So we calibrate trust on labelled data first, then deploy the same pipeline where no
  labels exist.

Recolour the embedding by the truth, and compute ARI for each clustering.

### The success: three behavioural families

`k = 3` scores **ARI ~ 0.74** — strong. To see *why*, we build a **cross-tab** (also called
a contingency table). How to read it:

- **Each row is one true class** (the held-aside label — M-dwarf flare, eclipsing binary,
  RR Lyrae, microlens, Mira).
- **Each column is one cluster** that k-means produced. The cluster numbers (`0`, `1`, `2`)
  are just arbitrary tags — k-means has no idea what the classes are called, so there is no
  reason cluster `0` should mean anything in particular. We only care about *which objects
  land together*, not the label on the box.
- **Each cell counts** how many objects of that true class fell into that cluster.

So a clean recovery looks like **one big number per row, in a different column for each
class** — every member of a class swept into the same box, and different classes in
different boxes. Let's look:

Read it off:

- one cluster is almost pure **M-dwarf flares** (eruptive, single-spike variability),
- one is almost pure **eclipsing binaries** (periodic dips),
- one collects the **pulsators** — RR Lyrae and *all 30 Miras* — together with the leftover
  microlenses and a minority of eclipsing binaries.

UL genuinely rediscovered the physical groupings *flaring / eclipsing / pulsating* with no
labels. That is a real result.

### The failure: pushing to k = 5

We have five named classes, so it is tempting to demand `k = 5`.
Look at what k-means actually did:

The two extra clusters do **not** go to the rare classes. Instead k-means **splits the
abundant M-dwarf flares (981 objects) into three pieces**, while RR Lyrae and Mira stay
glued together and the microlenses stay smeared across clusters.

**Why it fails — the mechanism.** k-means minimises total within-cluster squared distance,
and it implicitly assumes clusters that are *round and similarly sized*. The M-dwarf island
is large and elongated, so slicing it into thirds removes far more total variance than
isolating 30 Miras ever could. Given two spare clusters, the maths *always* prefers to
subdivide the biggest blob over carving out a tiny one. The rare classes simply do not have
enough points to pull a centroid of their own — Mira (n=30) never forms its own cluster at
any `k`. The same imbalance that we flagged in Part 1 is exactly what breaks the recovery
here.

And note: the silhouette in Part 5 was no higher at `k=3` than at `k=5`. Nothing internal to
the data told us `k=3` was the better answer — only the labels did.

## Part 7 — What we learned
